In [ ]:
# SAE-RSV: SAE-based Refined Steering Vector
# Using the unified steering module

import os
import torch
import numpy as np

from Steering import SteeringPipeline

# NOTE: Full SAE-RSV requires:
#   1. Feature descriptions from Neuronpedia
#   2. LLM for semantic relevance judgments (e.g., GPT-4)
# This demo uses simplified version without external API.

## 1. Initialize Pipeline

In [ ]:
# Create pipeline with SAE
pipeline = SteeringPipeline(
    model_name="google/gemma-2-2b",
    device="cuda:0",
    dtype=torch.bfloat16,
)

# Authenticate and load model (SAE methods need HookedSAETransformer)
pipeline.authenticate()
pipeline.load_model(use_sae_transformer=True)

In [ ]:
# Load SAE
TARGET_LAYER = 14

## 2. Load Dataset

In [ ]:
# Load contrastive dataset
DATASET_KEY = "sycophancy"

target_data, contrast_data = pipeline.load_train_data(
    dataset_name=DATASET_KEY,
    n_samples=100,
)

print(f"Loaded {len(target_data)} target samples and {len(contrast_data)} contrast samples")

## 3. Extract Refined Steering Vector

SAE-RSV process:
1. Compute original steering vector via CAA
2. Identify seed features (positive contribution in SAE space)
3. Classify features as noise/useful via semantic analysis
4. Refine vector using weighted combination

In [ ]:
# Extract refined steering vector
# NOTE: For full functionality, pass feature_descriptions parameter
refined_vector = pipeline.extract(
    method="SAE-RSV",
    target_data=target_data,
    contrast_data=contrast_data,
    layer=TARGET_LAYER,
    alpha1=1.0,  # Weight for original vector
    alpha2=0.5,  # Weight for noise subtraction
    alpha3=0.5,  # Weight for useful addition
    # feature_descriptions=neuronpedia_descriptions,  # Optional
)

print(f"Refined vector shape: {refined_vector.shape}")
if hasattr(pipeline.extractor, 'original_vector'):
    print(f"Original vector norm: {pipeline.extractor.original_vector.norm():.4f}")
    print(f"Refined vector norm: {refined_vector.norm():.4f}")

## 4. Create Steered Model and Generate

In [ ]:
# Setup SAE-RSV steered model
pipeline.steering(method="SAE-RSV", layer=TARGET_LAYER)

# Test prompt
TEST_PROMPT = "I think classical music is boring. What's your take on it?"

print(f"Prompt: {TEST_PROMPT}\n")
print("=" * 60)

print("\nBaseline (no steering):")
print(pipeline.generate(TEST_PROMPT, coeff=0.0, max_new_tokens=100, apply_steer=False))

print("\nSteered with refined vector (coeff=1.0):")
print(pipeline.generate(TEST_PROMPT, coeff=1.0, max_new_tokens=100))

## 5. Coefficient Sweep

In [ ]:
# Test with different coefficients
COEFFICIENTS = [-1.0, 0.0, 1.0, 2.0]

print(f"Testing with prompt: {TEST_PROMPT[:50]}...\n")
print("=" * 80)

for coeff in COEFFICIENTS:
    if coeff == 0.0:
        output = pipeline.generate(TEST_PROMPT, coeff=coeff, max_new_tokens=60, apply_steer=False)
    else:
        output = pipeline.generate(TEST_PROMPT, coeff=coeff, max_new_tokens=60)
    print(f"\nCoeff = {coeff:+.1f}:")
    print(output[:120] + "..." if len(output) > 120 else output)

## 6. Compare with Original CAA Vector

SAE-RSV should improve upon basic CAA by removing noise and adding relevant features.

In [ ]:
# Extract original CAA vector for comparison
from Steering import CAAExtractor, DenseSteerModel

# Create CAA extractor
caa_extractor = CAAExtractor(
    model=pipeline.model,
    layer=TARGET_LAYER,
    batch_size=16,
)

# Extract CAA vector
caa_vector = caa_extractor.extract(target_data, contrast_data)

# Compare norms and cosine similarity
cos_sim = torch.cosine_similarity(
    refined_vector.flatten().unsqueeze(0),
    caa_vector.flatten().unsqueeze(0),
).item()

print(f"CAA vector norm: {caa_vector.norm():.4f}")
print(f"SAE-RSV refined norm: {refined_vector.norm():.4f}")
print(f"Cosine similarity: {cos_sim:.4f}")